# Notebook 17 — Evaluation & Benchmarking

**Vision & 3D Mapping Workshop** | Block 7: Evaluation & Capstone

---

## What You'll Learn

How do we know if our SLAM/VO/depth system is any good? This notebook covers
**all standard evaluation metrics** used in the research community to benchmark
vision and 3D reconstruction systems. You will implement each metric from
scratch and understand the math behind it.

### Topics
1. **Trajectory evaluation**: ATE (Absolute Trajectory Error) with Umeyama Sim(3) alignment
2. **Relative Pose Error (RPE)**: local motion consistency
3. **Depth evaluation**: Eigen metrics (Abs Rel, Sq Rel, RMSE, $\delta$-thresholds)
4. **Scale-invariant depth evaluation**: aligning relative depth, SI-log loss
5. **3D reconstruction metrics**: Chamfer distance, accuracy, completeness, F-score
6. **Rendering metrics**: PSNR, SSIM (from scratch), LPIPS
7. **Standard benchmarks**: KITTI, TUM RGB-D, EuRoC, Replica, ScanNet, ETH3D

### Key Insight
Each metric captures a **different aspect** of system quality. ATE tells you about
global trajectory accuracy, RPE about local smoothness, depth metrics about
per-pixel prediction quality, and rendering metrics about visual fidelity.
A system can score well on one and poorly on another.

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 6)
np.set_printoptions(precision=4, suppress=True)

---
## 1. Absolute Trajectory Error (ATE)

### Umeyama Alignment

Before computing ATE, we must **align** the estimated trajectory to ground truth
using a similarity transformation (Sim(3): scale + rotation + translation).

Find $s, R, \mathbf{t}$ that minimise:

$$
J(s, R, \mathbf{t})
= \sum_{i=1}^{N} \bigl\| \mathbf{q}_i - (s\,R\,\mathbf{p}_i + \mathbf{t}) \bigr\|^2
$$

where $\mathbf{p}_i$ are the estimated positions and $\mathbf{q}_i$ are
the ground-truth positions.

### Umeyama Algorithm — Full SVD Derivation (1991)

**Step 1 — Eliminate translation by centring.**  Define centroids:

$$
\bar{\mathbf{p}} = \frac{1}{N}\sum_{i=1}^{N} \mathbf{p}_i, \qquad
\bar{\mathbf{q}} = \frac{1}{N}\sum_{i=1}^{N} \mathbf{q}_i
$$

and centred coordinates
$\hat{\mathbf{p}}_i = \mathbf{p}_i - \bar{\mathbf{p}}$,
$\hat{\mathbf{q}}_i = \mathbf{q}_i - \bar{\mathbf{q}}$.
Setting $\partial J / \partial \mathbf{t} = 0$ gives
$\mathbf{t} = \bar{\mathbf{q}} - s\,R\,\bar{\mathbf{p}}$,
so the translation is determined once $s$ and $R$ are known.

Substituting back, the objective reduces to centred coordinates:

$$
J(s, R)
= \sum_{i=1}^{N} \bigl\| \hat{\mathbf{q}}_i - s\,R\,\hat{\mathbf{p}}_i \bigr\|^2
$$

**Step 2 — Expand the squared norm.** Define the centred variances:

$$
\sigma_p^2 = \frac{1}{N}\sum_{i}\|\hat{\mathbf{p}}_i\|^2, \qquad
\sigma_q^2 = \frac{1}{N}\sum_{i}\|\hat{\mathbf{q}}_i\|^2
$$

Expanding $J$ and dividing by $N$:

$$
\frac{J}{N}
= \sigma_q^2 + s^2\sigma_p^2
  - 2s \cdot \frac{1}{N}\sum_{i} \hat{\mathbf{q}}_i^T R\,\hat{\mathbf{p}}_i
$$

The only term involving $R$ is the last one.  Maximising
$\sum_i \hat{\mathbf{q}}_i^T R\,\hat{\mathbf{p}}_i
= \text{tr}\!\bigl(R\,H^T\bigr)$ gives the optimal rotation, where
$H$ is the cross-covariance matrix.

**Step 3 — Cross-covariance and SVD.** Compute:

$$
H = \frac{1}{N}\sum_{i=1}^{N}
    \hat{\mathbf{p}}_i \, \hat{\mathbf{q}}_i^T
$$

Take the SVD: $H = U \Sigma V^T$.

**Step 4 — Optimal rotation.**  The problem
$\max_R \text{tr}(R\,H^T)$ subject to $R \in SO(3)$ has the
well-known solution (Arun et al. 1987):

$$
R = V \, S \, U^T, \qquad
S = \text{diag}(1, \ldots, 1, \det(V U^T))
$$

The diagonal matrix $S$ ensures $\det(R) = +1$ (proper rotation, not a
reflection).  If $\det(VU^T) = +1$, then $S = I$ and $R = VU^T$
directly.  If $\det(VU^T) = -1$, the last diagonal entry of $S$ becomes
$-1$, flipping the sign of the smallest singular component to avoid a
reflection.

**Step 5 — Optimal scale.**  Substituting the optimal $R$ back into
$\partial J / \partial s = 0$ and solving:

$$
s = \frac{\text{tr}(\Sigma\,S)}{N\,\sigma_p^2}
  = \frac{\sum_{j}\sigma_j \cdot S_{jj}}{N\,\sigma_p^2}
$$

where $\sigma_j$ are the singular values of $H$.  Intuitively, the scale
equals the sum of (sign-corrected) singular values divided by the
source-point variance.

**Step 6 — Translation.**

$$
\mathbf{t} = \bar{\mathbf{q}} - s\,R\,\bar{\mathbf{p}}
$$

### ATE

After alignment ($\mathbf{p}_i^{\text{aligned}} = s\,R\,\mathbf{p}_i + \mathbf{t}$):

$$
\text{ATE}_{\text{RMSE}} = \sqrt{\frac{1}{N} \sum_{i=1}^{N} \| \mathbf{p}_i^{\text{aligned}} - \mathbf{q}_i \|^2}
$$

In [ ]:
from src.eval import compute_ate, compute_rpe, compute_scale_error, align_trajectories_umeyama, plot_trajectory_comparison

# Generate synthetic ground truth trajectory (circle)
N = 100
t = np.linspace(0, 2*np.pi, N)
gt_positions = np.stack([5*np.cos(t), 5*np.sin(t), 0.5*np.sin(2*t)], axis=1)

# Simulate estimated trajectory with drift
drift = np.linspace(0, 0.3, N)[:, None] * np.array([1, 0.5, 0.2])
noise = np.random.normal(0, 0.05, gt_positions.shape)
est_positions = gt_positions + drift + noise

# Also apply a scale and rotation offset (simulating monocular VO)
scale = 0.95
theta = np.radians(3)
R_offset = np.array([[np.cos(theta), -np.sin(theta), 0],
                      [np.sin(theta), np.cos(theta), 0],
                      [0, 0, 1]])
est_positions = scale * (est_positions @ R_offset.T) + np.array([0.2, -0.1, 0.05])

# Build 4x4 pose matrices
gt_poses = [np.eye(4) for _ in range(N)]
est_poses = [np.eye(4) for _ in range(N)]
for i in range(N):
    gt_poses[i][:3, 3] = gt_positions[i]
    est_poses[i][:3, 3] = est_positions[i]

# Compute ATE
ate_rmse, ate_mean, ate_median, aligned, scale = compute_ate(est_poses, gt_poses)
print(f"ATE (after Umeyama alignment):")
print(f"  RMSE:   {ate_rmse:.4f} m")
print(f"  Mean:   {ate_mean:.4f} m")
print(f"  Median: {ate_median:.4f} m")

# Plot
plot_trajectory_comparison(est_poses, gt_poses, title="Trajectory Comparison")

### Scale Error (Monocular Systems)

For monocular VO/SLAM, the estimated trajectory has **unknown global scale** —
the system cannot distinguish a 1 m room from a 10 m room without metric
references (known object sizes, IMU, stereo baseline, etc.).

Umeyama alignment recovers the optimal scale $s^*$ that best maps the estimated
trajectory onto the ground truth. The **scale error** quantifies how far $s^*$
deviates from unity:

$$
e_s = |1 - s^*|
$$

- $e_s = 0$ → perfect scale recovery
- $e_s = 0.05$ → 5 % scale error

**Percentage scale error:** $|1 - s^*| \times 100\%$

**Why it matters:** Monocular VO/SLAM systems commonly exhibit **5–20 % scale
error** in the absence of metric references. Reporting this metric separately
from ATE isolates the scale failure mode — a system can have small ATE (after
alignment) yet large scale error, meaning it relies entirely on the alignment
step to fix scale.

In [ ]:
scale_err, scale_err_pct = compute_scale_error(scale)

print(f"Umeyama alignment scale s* = {scale:.4f}")
print(f"Scale error  |1 - s*|     = {scale_err:.4f}")
print(f"Scale error  (percentage) = {scale_err_pct:.2f}%")
print()
print("Interpretation:")
if scale_err_pct < 2:
    print("  Excellent — scale nearly perfect (<2%)")
elif scale_err_pct < 5:
    print("  Good — typical for systems with weak metric cues (2-5%)")
elif scale_err_pct < 15:
    print("  Moderate — common for pure monocular VO (5-15%)")
else:
    print("  Poor — significant scale drift (>15%)")

---
## 2. Relative Pose Error (RPE)

While ATE measures global accuracy, RPE measures **local consistency** —
how well the system tracks frame-to-frame motion.

### Derivation

Define the **relative motion** over a window of $\delta$ frames in
ground truth and estimate:

$$
\Delta T_{\text{gt},i}   = T_{\text{gt},i}^{-1}\, T_{\text{gt},i+\delta},
\qquad
\Delta T_{\text{est},i}  = T_{\text{est},i}^{-1}\, T_{\text{est},i+\delta}
$$

Each $\Delta T$ captures the ego-motion between frames $i$ and
$i{+}\delta$ expressed in the **local frame at time $i$**.

The RPE at step $i$ is the discrepancy between these two relative motions:

$$
E_i = \Delta T_{\text{gt},i}^{-1}\;\Delta T_{\text{est},i}
    = \bigl(T_{\text{gt},i}^{-1}\,T_{\text{gt},i+\delta}\bigr)^{-1}
      \bigl(T_{\text{est},i}^{-1}\,T_{\text{est},i+\delta}\bigr)
$$

### Why the double-inverse structure?

The formula composes four pose inverses/products because we must
**compare motions in a common local frame**.  Working in the ground-truth
local frame at time $i$:

1. $\Delta T_{\text{gt},i}^{-1}$ "undoes" the ground-truth motion,
   mapping frame $i{+}\delta$ back to frame $i$.
2. $\Delta T_{\text{est},i}$ then applies the estimated motion.
3. If the estimate is perfect, $E_i = I$.  Any deviation from the
   identity is the local tracking error.

Comparing motions in a local frame (rather than the global frame) is
critical because it makes RPE **invariant to the global alignment** used
for ATE — RPE isolates drift rate from global offset.

### Decomposition

$E_i$ is decomposed into translation and rotation components:

- **Translation RPE**: $\| \text{trans}(E_i) \|$ — local position drift (metres)
- **Rotation RPE**: $\angle(E_i) = \arccos\!\bigl(\frac{\text{tr}(R_{E_i}) - 1}{2}\bigr)$ — local orientation drift (radians)

Aggregate statistics (RMSE, mean, median) are computed over all $i$:

$$
\text{RPE}_{\text{trans, RMSE}} = \sqrt{\frac{1}{M}\sum_{i=1}^{M}\|\text{trans}(E_i)\|^2}
$$

In [ ]:
rpe_trans_rmse, rpe_trans_mean, rpe_rot_rmse, rpe_rot_mean = compute_rpe(est_poses, gt_poses, delta=1)
print(f"RPE (δ=1):")
print(f"  Translation RMSE: {rpe_trans_rmse:.4f} m")
print(f"  Translation Mean: {rpe_trans_mean:.4f} m")
print(f"  Rotation RMSE:    {np.degrees(rpe_rot_rmse):.4f} deg")
print(f"  Rotation Mean:    {np.degrees(rpe_rot_mean):.4f} deg")

---
## 3. Depth Evaluation Metrics

Standard metrics from Eigen et al. (2014), with $d$ = predicted depth,
$d^*$ = ground-truth depth, and $N$ = number of valid pixels:

| Metric | Formula | Interpretation |
|--------|---------|----------------|
| Abs Rel | $\frac{1}{N}\sum \frac{|d - d^*|}{d^*}$ | Relative error |
| Sq Rel | $\frac{1}{N}\sum \frac{(d - d^*)^2}{d^*}$ | Penalises large errors quadratically |
| RMSE | $\sqrt{\frac{1}{N}\sum(d - d^*)^2}$ | Absolute error in metres |
| RMSE log | $\sqrt{\frac{1}{N}\sum(\log d - \log d^*)^2}$ | Scale-aware error |
| $\delta_1$ | $\%\;\max(d/d^*,\; d^*/d) < 1.25$ | Accuracy threshold |
| $\delta_2$ | $< 1.25^2 = 1.5625$ | Looser threshold |
| $\delta_3$ | $< 1.25^3 \approx 1.953$ | Loosest threshold |

### Why 1.25 and three thresholds?

The $\delta$ thresholds measure the fraction of pixels whose predicted
depth is within a multiplicative factor of the ground truth.  The
threshold $\max(d/d^*,\, d^*/d) < \tau$ is equivalent to requiring:

$$
\frac{1}{\tau} < \frac{d}{d^*} < \tau
$$

**Why $\tau = 1.25$?**  A 25 % relative error is the boundary between
"usable for downstream 3-D tasks" (obstacle avoidance, reconstruction)
and "unreliable."  Eigen et al. chose it empirically as the strictest
threshold at which strong monocular depth models achieve $>80\%$ accuracy,
providing a meaningful spread across methods.

**Three levels** ($1.25,\; 1.25^2,\; 1.25^3$) form a geometric
progression that reveals the error distribution:

| Threshold | Max relative error | Typical "good" score |
|-----------|--------------------|----------------------|
| $\delta_1 < 1.25$   | ±25 % | > 0.95 |
| $\delta_2 < 1.5625$ | ±56 % | > 0.99 |
| $\delta_3 < 1.953$  | ±95 % | > 0.998 |

If $\delta_1$ is high but $\delta_3$ is also high, errors are small.
If $\delta_1$ is low but $\delta_3$ is high, the model has a long tail
of moderate errors.  If $\delta_3$ is low, the model produces
catastrophic outliers (depth off by $> 2\times$).

In [ ]:
from src.eval import compute_depth_metrics

# Synthetic depth prediction with noise
H, W = 120, 160
gt_depth = np.random.uniform(1, 10, (H, W)).astype(np.float32)
pred_depth = gt_depth * (1 + np.random.normal(0, 0.08, gt_depth.shape)).astype(np.float32)
pred_depth = np.clip(pred_depth, 0.1, 100)

metrics = compute_depth_metrics(pred_depth, gt_depth)

print("Depth Evaluation Metrics:")
print(f"  Abs Rel:  {metrics['abs_rel']:.4f}")
print(f"  Sq Rel:   {metrics['sq_rel']:.4f}")
print(f"  RMSE:     {metrics['rmse']:.4f}")
print(f"  δ < 1.25:  {metrics['delta_1']:.4f}")
print(f"  δ < 1.25²: {metrics['delta_2']:.4f}")
print(f"  δ < 1.25³: {metrics['delta_3']:.4f}")

---
## 4. 3D Reconstruction Metrics

### Chamfer Distance

$$
d_{\text{CD}}(S_1, S_2) = \frac{1}{|S_1|} \sum_{x \in S_1} \min_{y \in S_2} \|x - y\|
+ \frac{1}{|S_2|} \sum_{y \in S_2} \min_{x \in S_1} \|y - x\|
$$

- First term = **accuracy** (how close reconstruction is to GT)
- Second term = **completeness** (how well GT is covered)

### F-Score at Threshold $\tau$

$$
\text{Precision} = \frac{\#\{x \in S_1 : \min_y \|x-y\| < \tau\}}{|S_1|}, \quad
\text{Recall} = \frac{\#\{y \in S_2 : \min_x \|x-y\| < \tau\}}{|S_2|}
$$

$$
F = \frac{2 \cdot \text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}
$$

In [ ]:
from scipy.spatial import KDTree

def chamfer_distance(pts1, pts2):
    """Chamfer distance between two point clouds."""
    tree1 = KDTree(pts1)
    tree2 = KDTree(pts2)
    d1, _ = tree2.query(pts1)
    d2, _ = tree1.query(pts2)
    accuracy = d1.mean()
    completeness = d2.mean()
    return accuracy + completeness, accuracy, completeness

def f_score(pts1, pts2, tau=0.05):
    """F-score at threshold tau."""
    tree1 = KDTree(pts1)
    tree2 = KDTree(pts2)
    d1, _ = tree2.query(pts1)
    d2, _ = tree1.query(pts2)
    precision = (d1 < tau).mean()
    recall = (d2 < tau).mean()
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)

# Synthetic point clouds
gt_pts = np.random.randn(1000, 3) * 2
noisy_pts = gt_pts + np.random.normal(0, 0.03, gt_pts.shape)
missing_pts = noisy_pts[np.random.choice(len(noisy_pts), 800, replace=False)]  # 80% completeness

cd, acc, comp = chamfer_distance(missing_pts, gt_pts)
f = f_score(missing_pts, gt_pts, tau=0.1)

print(f"Chamfer Distance: {cd:.4f}")
print(f"  Accuracy:       {acc:.4f}")
print(f"  Completeness:   {comp:.4f}")
print(f"F-Score (τ=0.1):  {f:.4f}")

---
## 5. Scale-Invariant Depth Evaluation

### The Scale/Shift Problem

Monocular depth networks predict *relative* depth — the output is related to
true depth by an unknown scale $\alpha$ and shift $\beta$:

$$
d_{\text{predicted}} = \alpha \cdot d_{\text{true}} + \beta
$$

Before computing metrics, we must **align** the prediction to ground truth.

### Least-Squares Alignment

Find $\alpha, \beta$ minimising:

$$
\min_{\alpha, \beta} \sum_{i} \bigl(d_i^{\text{gt}} - \alpha \cdot d_i^{\text{pred}} - \beta\bigr)^2
$$

This is a simple linear regression.  Writing $A$ for the design matrix and
$\mathbf{g}$ for the ground-truth vector:

$$
A = \begin{pmatrix}
      d_1^{\text{pred}} & 1 \\
      \vdots & \vdots \\
      d_N^{\text{pred}} & 1
    \end{pmatrix}, \qquad
\mathbf{g} = \begin{pmatrix} d_1^{\text{gt}} \\ \vdots \\ d_N^{\text{gt}} \end{pmatrix}
$$

The least-squares solution is:

$$
\begin{pmatrix} \alpha \\ \beta \end{pmatrix}
= (A^T A)^{-1} A^T \mathbf{g}
$$

The aligned prediction is then
$d_i^{\text{aligned}} = \alpha \cdot d_i^{\text{pred}} + \beta \approx d_i^{\text{gt}}$.

### Scale-Invariant Loss (Eigen et al. 2014)

$$
\mathcal{L}_{\text{SI}} = \frac{1}{N} \sum_i g_i^2 - \frac{\lambda}{N^2} \left(\sum_i g_i\right)^2
$$

where $g_i = \log d_i - \log d_i^*$. The second term removes the global scale component.

In [ ]:
def align_depth_least_squares(pred, gt, valid_mask=None):
    """Align predicted depth to GT via least-squares (find scale α and shift β).
    
    Solves: min_α,β Σ(gt_i - α·pred_i - β)²  →  aligned = α·pred + β ≈ gt
    """
    if valid_mask is None:
        valid_mask = (gt > 0) & (pred > 0)
    p = pred[valid_mask].ravel()
    g = gt[valid_mask].ravel()
    
    # Linear regression: [pred, 1] @ [α, β]^T = gt
    A = np.stack([p, np.ones_like(p)], axis=1)
    result = np.linalg.lstsq(A, g, rcond=None)
    alpha, beta = result[0]
    
    aligned = alpha * pred.astype(np.float64) + beta
    return aligned, alpha, beta

def scale_invariant_loss(pred, gt, valid_mask=None, lam=0.5):
    """Scale-invariant log loss (Eigen et al. 2014)."""
    if valid_mask is None:
        valid_mask = (gt > 0) & (pred > 0)
    g = np.log(pred[valid_mask]) - np.log(gt[valid_mask])
    N = len(g)
    return float(np.mean(g**2) - lam * (np.sum(g)**2) / (N**2))

# Demonstrate alignment
gt_depth_demo = np.random.uniform(1, 10, (120, 160)).astype(np.float32)
# Simulate a network that predicts depth with wrong scale and shift
pred_raw = 0.3 * gt_depth_demo + 1.5 + np.random.normal(0, 0.2, gt_depth_demo.shape).astype(np.float32)

aligned, alpha, beta = align_depth_least_squares(pred_raw, gt_depth_demo)

metrics_raw = compute_depth_metrics(pred_raw, gt_depth_demo)
metrics_aligned = compute_depth_metrics(aligned, gt_depth_demo)

print(f"Alignment: α = {alpha:.4f}, β = {beta:.4f}")
print(f"\nBefore alignment:")
print(f"  Abs Rel: {metrics_raw['abs_rel']:.4f}")
print(f"  δ₁:     {metrics_raw['delta_1']:.4f}")
print(f"\nAfter alignment:")
print(f"  Abs Rel: {metrics_aligned['abs_rel']:.4f}")
print(f"  δ₁:     {metrics_aligned['delta_1']:.4f}")
print(f"\nScale-invariant loss: {scale_invariant_loss(pred_raw, gt_depth_demo):.6f}")

---
## 6. PSNR and SSIM Implementation

### PSNR

$$
\text{PSNR} = 10 \cdot \log_{10}\!\left(\frac{\text{MAX}^2}{\text{MSE}}\right)
= 20 \cdot \log_{10}\!\left(\frac{\text{MAX}}{\sqrt{\text{MSE}}}\right)
$$

For images with pixel values in $[0, 1]$, $\text{MAX} = 1$.

### SSIM (Wang et al. 2004)

The Structural Similarity Index decomposes image similarity into three
components — luminance $l$, contrast $c$, and structure $s$:

$$
\text{SSIM}(x, y) = \underbrace{\frac{2\mu_x\mu_y + C_1}{\mu_x^2 + \mu_y^2 + C_1}}_{l(x,y)}
\cdot \underbrace{\frac{2\sigma_x\sigma_y + C_2}{\sigma_x^2 + \sigma_y^2 + C_2}}_{c(x,y)}
\cdot \underbrace{\frac{\sigma_{xy} + C_3}{\sigma_x\sigma_y + C_3}}_{s(x,y)}
$$

**Stability constants:**

$$
C_1 = (k_1 L)^2, \quad C_2 = (k_2 L)^2, \quad C_3 = C_2 / 2
$$

with $k_1 = 0.01$, $k_2 = 0.03$, and $L$ the dynamic range of the pixel
values ($L = 255$ for 8-bit images, $L = 1$ for $[0,1]$-normalised).
These constants prevent division-by-zero instability in dark or flat
regions.

**Simplified two-factor form.** Setting $C_3 = C_2/2$ and multiplying the
contrast and structure terms:

$$
c(x,y) \cdot s(x,y)
= \frac{2\sigma_x\sigma_y + C_2}{\sigma_x^2 + \sigma_y^2 + C_2}
  \cdot \frac{\sigma_{xy} + C_2/2}{\sigma_x\sigma_y + C_2/2}
= \frac{2\sigma_{xy} + C_2}{\sigma_x^2 + \sigma_y^2 + C_2}
$$

(The identity follows because
$\frac{2\sigma_x\sigma_y}{\sigma_x\sigma_y} \cdot \frac{\sigma_{xy}}{\sigma_{xy}} = 2$, and the additive constants absorb consistently.)

This gives the **standard implementation formula**:

$$
\boxed{
\text{SSIM}(x, y) = \frac{(2\mu_x\mu_y + C_1)(2\sigma_{xy} + C_2)}
                         {(\mu_x^2 + \mu_y^2 + C_1)(\sigma_x^2 + \sigma_y^2 + C_2)}
}
$$

This is the form used in practice and in the code cell below.  Statistics
$\mu$, $\sigma^2$, $\sigma_{xy}$ are computed within an $11 \times 11$
Gaussian window ($\sigma_{\text{window}} = 1.5$) centred at each pixel,
yielding a per-pixel SSIM map whose mean is the reported scalar.

In [ ]:
from scipy.ndimage import gaussian_filter

def compute_psnr(img1, img2, max_val=1.0):
    """Peak Signal-to-Noise Ratio."""
    mse = np.mean((img1.astype(np.float64) - img2.astype(np.float64))**2)
    if mse == 0:
        return float('inf')
    return float(10 * np.log10(max_val**2 / mse))

def compute_ssim(img1, img2, window_size=11, max_val=1.0):
    """Structural Similarity Index (Wang et al. 2004).
    
    Uses an 11x11 Gaussian window with sigma=1.5 as specified
    in the original paper.
    """
    C1 = (0.01 * max_val)**2
    C2 = (0.03 * max_val)**2
    sigma = 1.5
    
    img1 = img1.astype(np.float64)
    img2 = img2.astype(np.float64)
    
    mu1 = gaussian_filter(img1, sigma)
    mu2 = gaussian_filter(img2, sigma)
    
    mu1_sq = mu1 * mu1
    mu2_sq = mu2 * mu2
    mu12 = mu1 * mu2
    
    sigma1_sq = gaussian_filter(img1 * img1, sigma) - mu1_sq
    sigma2_sq = gaussian_filter(img2 * img2, sigma) - mu2_sq
    sigma12 = gaussian_filter(img1 * img2, sigma) - mu12
    
    numerator = (2 * mu12 + C1) * (2 * sigma12 + C2)
    denominator = (mu1_sq + mu2_sq + C1) * (sigma1_sq + sigma2_sq + C2)
    
    ssim_map = numerator / denominator
    return float(np.mean(ssim_map))

# Demonstrate on synthetic images
img_gt = np.random.rand(128, 128).astype(np.float64)
img_noisy = np.clip(img_gt + np.random.normal(0, 0.05, img_gt.shape), 0, 1)
img_blurry = gaussian_filter(img_gt, 2.0)

psnr_noisy = compute_psnr(img_gt, img_noisy)
psnr_blurry = compute_psnr(img_gt, img_blurry)
ssim_noisy = compute_ssim(img_gt, img_noisy)
ssim_blurry = compute_ssim(img_gt, img_blurry)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(img_gt, cmap='gray')
axes[0].set_title('Ground Truth')
axes[1].imshow(img_noisy, cmap='gray')
axes[1].set_title(f'Noisy\nPSNR={psnr_noisy:.1f}dB, SSIM={ssim_noisy:.3f}')
axes[2].imshow(img_blurry, cmap='gray')
axes[2].set_title(f'Blurry\nPSNR={psnr_blurry:.1f}dB, SSIM={ssim_blurry:.3f}')
for ax in axes:
    ax.axis('off')
plt.suptitle('PSNR vs SSIM — Noise vs Blur', fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Noisy:  PSNR = {psnr_noisy:.2f} dB, SSIM = {ssim_noisy:.4f}")
print(f"Blurry: PSNR = {psnr_blurry:.2f} dB, SSIM = {ssim_blurry:.4f}")
print("\nNote: SSIM penalises blur more than noise — closer to human perception.")

---
## 7. LPIPS — Learned Perceptual Image Patch Similarity

PSNR and SSIM are pixel-level metrics. LPIPS measures perceptual similarity
using deep features from a pretrained network (AlexNet/VGG).

### Intuition

Two images can have identical PSNR but look very different to humans (e.g.,
a slightly shifted image vs. a blurred one). LPIPS correlates better with
human judgment because it compares images in learned feature space.

### Math

Given images $x, y$ and a pretrained network $\phi$, extract features at
layers $l$:

$$
\text{LPIPS}(x, y) = \sum_l \frac{1}{H_l W_l} \sum_{h,w}
\| w_l \odot (\hat{\phi}_l^x(h,w) - \hat{\phi}_l^y(h,w)) \|_2^2
$$

where $\hat{\phi}_l$ are channel-normalised features and $w_l$ are learned
linear weights. **Lower is better** (opposite of PSNR/SSIM).

### Typical Ranges for NeRF/3DGS

| Metric | Poor | Good | Excellent |
|--------|------|------|-----------|
| PSNR ↑ | <25 dB | 28-32 dB | >35 dB |
| SSIM ↑ | <0.85 | 0.90-0.95 | >0.97 |
| LPIPS ↓ | >0.20 | 0.10-0.15 | <0.05 |

> **Note**: LPIPS requires `torch` + `lpips` package. We discuss it here
> for completeness but don't run it to keep this notebook CPU-friendly.

---

## 8. Standard Benchmarks

Understanding **which benchmark to use** is as important as knowing the metrics.

| Benchmark | Domain | Sensors | Key Metrics | Sequences | Notes |
|-----------|--------|---------|-------------|-----------|-------|
| **KITTI** | Driving | Stereo + LiDAR + GPS/INS | ATE, depth, flow | 22 sequences | The most cited benchmark for VO/SLAM |
| **TUM RGB-D** | Indoor | RGB-D (Kinect) | ATE, RPE | 39 sequences | Standard for RGB-D SLAM |
| **EuRoC MAV** | Industrial | Stereo + IMU | ATE | 11 sequences | Standard for VIO on drones |
| **Replica** | Synthetic | RGB-D | PSNR, SSIM, LPIPS, depth | 18 rooms | Standard for neural SLAM (SplaTAM, MonoGS) |
| **ScanNet** | Indoor | RGB-D | Segmentation, depth, pose | 1513 scenes | Large-scale indoor understanding |
| **ETH3D** | Mixed | Multi-view | F-score, accuracy, completeness | 25 scenes | Multi-view stereo benchmark |
| **Aachen Day-Night** | Outdoor | Monocular | Localisation accuracy | 6km² city | Visual localisation benchmark |

### Key Benchmark Insights

- **KITTI** uses special "sequence-level" ATE: evaluate on sequences 00-10 (with GT),
  report percentage of correctly aligned sub-paths at 100m, 200m, ..., 800m distances
- **TUM RGB-D** pioneered the `evo` evaluation tool (`evo_ape`, `evo_rpe`)
- **EuRoC** includes IMU data at 200Hz — essential for testing VIO
- **Replica** has pixel-perfect rendering ground truth — the gold standard for
  comparing neural vs classical reconstruction

---

## 9. Exercises

### Exercise 9.1: Compare Depth Alignment Methods
Try median scaling vs. least-squares alignment on the same predicted depth map.
When does each method work better?

### Exercise 9.2: ATE vs RPE Analysis
A system with low ATE but high RPE has good global but poor local accuracy (lucky
cancellation of errors). Generate such a trajectory and observe.

### Exercise 9.3: Metric Sensitivity
How do depth metrics change as you add increasing noise? Plot Abs Rel, RMSE,
and $\delta_1$ as a function of noise standard deviation.

---

## Key Takeaways

1. **ATE** is the standard for SLAM trajectory evaluation — always report RMSE after
   Umeyama alignment
2. **RPE** captures local drift rate — more informative for VO systems
3. **Depth metrics** should always include both error metrics (Abs Rel, RMSE) and
   accuracy metrics ($\delta_1$)
4. **Scale-invariant** evaluation is critical for monocular depth (networks predict
   relative, not absolute depth)
5. **SSIM > PSNR** for perceptual quality; **LPIPS** correlates best with human judgment
6. **F-score** is the single best metric for 3D reconstruction quality (balances
   accuracy and completeness)
7. Always report on **standard benchmarks** so your results are comparable

**Next**: Notebook 18 — Full Pipeline

In [ ]:
# Exercise 9.3: Metric Sensitivity Analysis
# How do depth metrics degrade as noise increases?

noise_levels = np.linspace(0.01, 0.30, 20)
abs_rels = []
rmses = []
delta1s = []

gt_ex = np.random.uniform(1, 10, (120, 160)).astype(np.float32)

for sigma in noise_levels:
    pred_ex = gt_ex * (1 + np.random.normal(0, sigma, gt_ex.shape)).astype(np.float32)
    pred_ex = np.clip(pred_ex, 0.1, 100)
    m = compute_depth_metrics(pred_ex, gt_ex)
    abs_rels.append(m['abs_rel'])
    rmses.append(m['rmse'])
    delta1s.append(m['delta_1'])

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(noise_levels * 100, abs_rels, 'b-o', markersize=3)
axes[0].set_xlabel('Noise σ (%)')
axes[0].set_ylabel('Abs Rel')
axes[0].set_title('Abs Rel vs Noise')
axes[0].grid(True, alpha=0.3)

axes[1].plot(noise_levels * 100, rmses, 'r-o', markersize=3)
axes[1].set_xlabel('Noise σ (%)')
axes[1].set_ylabel('RMSE (m)')
axes[1].set_title('RMSE vs Noise')
axes[1].grid(True, alpha=0.3)

axes[2].plot(noise_levels * 100, delta1s, 'g-o', markersize=3)
axes[2].set_xlabel('Noise σ (%)')
axes[2].set_ylabel('δ₁')
axes[2].set_title('δ₁ Threshold Accuracy vs Noise')
axes[2].axhline(y=0.95, color='k', linestyle='--', alpha=0.3, label='95% target')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.suptitle('Depth Metric Sensitivity to Noise', fontweight='bold')
plt.tight_layout()
plt.show()

print("Observation: Abs Rel grows linearly with noise, δ₁ degrades gracefully")
print("until ~15% noise, then drops sharply. RMSE grows roughly proportional to noise × mean depth.")

# 3D surface plot: depth error as a function of ground truth depth and noise
fig = plt.figure(figsize=(14, 5))

ax3d = fig.add_subplot(1, 2, 1, projection='3d')
noise_grid = np.linspace(0.01, 0.30, 20)
depth_grid = np.linspace(1, 10, 20)
NG, DG = np.meshgrid(noise_grid, depth_grid)
abs_rel_surface = NG  # Abs Rel ≈ σ for multiplicative noise
ax3d.plot_surface(NG * 100, DG, abs_rel_surface, cmap='viridis', alpha=0.8, edgecolor='none')
ax3d.set_xlabel('Noise σ [%]')
ax3d.set_ylabel('GT Depth [m]')
ax3d.set_zlabel('Abs Rel')
ax3d.set_title('Abs Rel vs Noise × Depth')
ax3d.view_init(elev=25, azim=-45)

ax3d2 = fig.add_subplot(1, 2, 2, projection='3d')
rmse_surface = NG * DG  # RMSE ≈ σ × depth for multiplicative noise
ax3d2.plot_surface(NG * 100, DG, rmse_surface, cmap='plasma', alpha=0.8, edgecolor='none')
ax3d2.set_xlabel('Noise σ [%]')
ax3d2.set_ylabel('GT Depth [m]')
ax3d2.set_zlabel('RMSE [m]')
ax3d2.set_title('RMSE vs Noise × Depth')
ax3d2.view_init(elev=25, azim=-45)

plt.suptitle('Depth Error Dependence on Noise and Distance', fontweight='bold')
plt.tight_layout()
plt.show()

print("Key insight: RMSE grows with both noise AND depth (multiplicative error model),")
print("while Abs Rel is independent of depth — it measures relative accuracy.")

In [ ]:
# Exercise 9.1: Median Scaling vs Least-Squares Alignment

def align_depth_median(pred, gt, valid_mask=None):
    """Median scaling: scale = median(gt) / median(pred)."""
    if valid_mask is None:
        valid_mask = (gt > 0) & (pred > 0)
    scale = np.median(gt[valid_mask]) / np.median(pred[valid_mask])
    return pred * scale, scale

# Case 1: Pure scale offset (median scaling should work well)
gt_case1 = np.random.uniform(2, 8, (120, 160)).astype(np.float32)
pred_case1 = gt_case1 * 0.6 + np.random.normal(0, 0.1, gt_case1.shape).astype(np.float32)

aligned_median, s_med = align_depth_median(pred_case1, gt_case1)
aligned_ls, alpha_ls, beta_ls = align_depth_least_squares(pred_case1, gt_case1)

m_raw = compute_depth_metrics(pred_case1, gt_case1)
m_med = compute_depth_metrics(aligned_median, gt_case1)
m_ls = compute_depth_metrics(aligned_ls, gt_case1)

print("Case 1: Scale + shift offset")
print(f"  Raw:    Abs Rel = {m_raw['abs_rel']:.4f}, δ₁ = {m_raw['delta_1']:.4f}")
print(f"  Median: Abs Rel = {m_med['abs_rel']:.4f}, δ₁ = {m_med['delta_1']:.4f}  (scale = {s_med:.3f})")
print(f"  LS:     Abs Rel = {m_ls['abs_rel']:.4f}, δ₁ = {m_ls['delta_1']:.4f}  (α = {alpha_ls:.3f}, β = {beta_ls:.3f})")
print()
print("Observation: LS handles scale+shift jointly, so it wins when both are present.")
print("Median scaling only corrects scale, not shift — use it when the shift is ~0.")

In [ ]:
# Exercise 9.2: ATE vs RPE Analysis

import numpy as np

def generate_low_ate_high_rpe(n_poses=100):
    """Trajectory where ATE is low but RPE is high.

    High-frequency zigzag cancels out globally (low ATE after alignment)
    but creates large frame-to-frame deviations (high RPE).
    """
    gt_pos = np.zeros((n_poses, 3))
    est_pos = np.zeros((n_poses, 3))
    for i in range(1, n_poses):
        gt_pos[i] = gt_pos[i-1] + np.array([0.1, 0, 0])
        est_pos[i] = gt_pos[i] + np.array([0, 0.08 * (-1)**i, 0])
    return gt_pos, est_pos

def generate_high_ate_low_rpe(n_poses=100):
    """Trajectory where ATE is high but RPE is low.

    Smooth, gradually increasing lateral drift accumulates into large
    global error (high ATE) that Sim(3) alignment cannot remove, but
    each individual step is nearly correct (low RPE).
    """
    gt_pos = np.zeros((n_poses, 3))
    est_pos = np.zeros((n_poses, 3))
    step_gt = np.array([0.1, 0.0, 0.0])
    for i in range(1, n_poses):
        gt_pos[i] = gt_pos[i-1] + step_gt
        lateral_bias = np.array([0.0, 0.03 * (i / n_poses), 0.0])
        est_pos[i] = est_pos[i-1] + step_gt + lateral_bias
    return gt_pos, est_pos

def poses_from_positions(positions):
    return [np.block([[np.eye(3), p.reshape(3,1)],
                      [np.zeros((1,3)), np.ones((1,1))]]) for p in positions]

# Case A: Low ATE, High RPE (zigzag)
gt_a, est_a = generate_low_ate_high_rpe()
ate_a = compute_ate(poses_from_positions(est_a), poses_from_positions(gt_a))
rpe_a = compute_rpe(poses_from_positions(est_a), poses_from_positions(gt_a), delta=1)

# Case B: High ATE, Low RPE (smooth drift)
gt_b, est_b = generate_high_ate_low_rpe()
ate_b = compute_ate(poses_from_positions(est_b), poses_from_positions(gt_b))
rpe_b = compute_rpe(poses_from_positions(est_b), poses_from_positions(gt_b), delta=1)

print("Exercise 9.2 — ATE vs RPE Contrast")
print("=" * 55)
print(f"{'Case':<25s} {'ATE RMSE':>10s} {'RPE trans':>10s}")
print("-" * 55)
print(f"{'A: zigzag (low ATE)':<25s} {ate_a[0]:>10.4f} {rpe_a[0]:>10.4f}")
print(f"{'B: drift  (high ATE)':<25s} {ate_b[0]:>10.4f} {rpe_b[0]:>10.4f}")
print()
print("Case A: Zigzag oscillations cancel globally → low ATE,")
print("        but each step deviates → high RPE.")
print("Case B: Smooth bias accumulates → high ATE,")
print("        but each step is nearly correct → low RPE.")

fig = plt.figure(figsize=(18, 10))

# Top row: 2D trajectory comparisons
ax1 = fig.add_subplot(2, 2, 1)
ax1.plot(gt_a[:, 0], gt_a[:, 1], "k-", lw=2, label="GT")
ax1.plot(est_a[:, 0], est_a[:, 1], "r-", alpha=0.7, label="Estimate")
for idx in range(0, len(gt_a), 5):
    ax1.annotate('', xy=est_a[idx,:2], xytext=gt_a[idx,:2],
                 arrowprops=dict(arrowstyle='->', color='red', alpha=0.3, lw=0.5))
ax1.set_title(f"Case A: Low ATE ({ate_a[0]:.3f}), High RPE ({rpe_a[0]:.3f})")
ax1.legend(fontsize=9); ax1.set_aspect("equal"); ax1.grid(True, alpha=0.3)
ax1.set_xlabel("X [m]"); ax1.set_ylabel("Y [m]")

ax2 = fig.add_subplot(2, 2, 2)
ax2.plot(gt_b[:, 0], gt_b[:, 1], "k-", lw=2, label="GT")
ax2.plot(est_b[:, 0], est_b[:, 1], "b-", alpha=0.7, label="Estimate")
for idx in range(0, len(gt_b), 5):
    ax2.annotate('', xy=est_b[idx,:2], xytext=gt_b[idx,:2],
                 arrowprops=dict(arrowstyle='->', color='blue', alpha=0.3, lw=0.5))
ax2.set_title(f"Case B: High ATE ({ate_b[0]:.3f}), Low RPE ({rpe_b[0]:.3f})")
ax2.legend(fontsize=9); ax2.set_aspect("equal"); ax2.grid(True, alpha=0.3)
ax2.set_xlabel("X [m]"); ax2.set_ylabel("Y [m]")

# Bottom left: 3D view of both cases
ax3 = fig.add_subplot(2, 2, 3, projection='3d')
ax3.plot(gt_a[:, 0], gt_a[:, 1], gt_a[:, 2], 'k-', lw=2, label='GT')
ax3.plot(est_a[:, 0], est_a[:, 1], est_a[:, 2], 'r-', alpha=0.6, label='Case A (zigzag)')
ax3.plot(est_b[:, 0], est_b[:, 1], est_b[:, 2], 'b-', alpha=0.6, label='Case B (drift)')
ax3.scatter(*gt_a[0], c='lime', s=80, marker='o', edgecolors='k', zorder=10)
ax3.scatter(*gt_a[-1], c='red', s=80, marker='X', edgecolors='k', zorder=10)
ax3.set_xlabel('X [m]'); ax3.set_ylabel('Y [m]'); ax3.set_zlabel('Z [m]')
ax3.set_title('3D Trajectory View (both cases)')
ax3.legend(fontsize=8); ax3.view_init(elev=25, azim=-45)

# Bottom right: error magnitude over time
ax4 = fig.add_subplot(2, 2, 4)
err_a = np.linalg.norm(est_a - gt_a, axis=1)
err_b = np.linalg.norm(est_b - gt_b, axis=1)
ax4.plot(err_a, 'r-', lw=1.5, label=f'Case A — ATE={ate_a[0]:.3f}')
ax4.plot(err_b, 'b-', lw=1.5, label=f'Case B — ATE={ate_b[0]:.3f}')
ax4.axhline(ate_a[0], color='r', ls='--', alpha=0.4)
ax4.axhline(ate_b[0], color='b', ls='--', alpha=0.4)
ax4.set_xlabel('Pose index'); ax4.set_ylabel('Position error [m]')
ax4.set_title('Per-Pose Error Magnitude')
ax4.legend(fontsize=9); ax4.grid(True, alpha=0.3)

plt.suptitle("ATE vs RPE — Different Failure Modes", fontweight="bold", fontsize=13)
plt.tight_layout()
plt.show()

## Summary

### Key Metrics Reference

| Domain | Metric | What It Measures |
|--------|--------|-----------------|
| Trajectory | ATE (RMSE) | Global position accuracy after Sim(3) alignment |
| Trajectory | Scale error | Monocular scale deviation $|1 - s^*|$ from alignment |
| Trajectory | RPE (trans/rot) | Local motion consistency (frame-to-frame drift) |
| Depth | Abs Rel | Mean relative depth error |
| Depth | δ < 1.25 | Fraction within 25% of ground truth |
| Depth | SI-log | Scale-invariant quality (for relative depth) |
| Reconstruction | Chamfer | Bidirectional mean distance to ground truth |
| Reconstruction | F-score | Harmonic mean of accuracy and completeness |
| Rendering | PSNR | Peak signal-to-noise ratio (pixel-level) |
| Rendering | SSIM | Structural similarity (perceptual) |
| Rendering | LPIPS | Learned perceptual distance (deep features) |

### Key Takeaways

- **Umeyama alignment** (Sim(3)) is essential for monocular systems — without it, ATE is meaningless
- Scale-invariant metrics allow fair comparison between relative and metric depth methods
- PSNR/SSIM measure pixel-level quality; LPIPS better correlates with human perception
- Different benchmarks stress different failure modes: KITTI = outdoor driving, TUM = indoor, EuRoC = drone

### What's Next

Notebook 18 integrates all components into a complete pipeline and evaluates it using these metrics.